### Student Perfomance

Notebook para experimentar con este problema cargando datos desde s3 y registrando en mlflow

In [1]:
# imports
import optuna
import pandas as pd
import mlflow
import sklearn
import awswrangler as wr
from dotenv import load_dotenv
import os
from pathlib import Path
from helpers.s3_helpers import get_latest_s3_folder
from helpers.model_helpers import logistic_regression_model_train_log
import sweetviz as sv
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
from mlflow import MlflowClient
from mlflow.exceptions import RestException

/home/nacho/Documents/CEIA/amq2-service-ml/notebooks/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load env variables
load_dotenv()

True

In [3]:
# connect to mlflow 
mlflow.set_tracking_uri("http://localhost:5001")

In [4]:
# Use most recent data
latest_path = get_latest_s3_folder(
    'data',
    'student_performance',
    '%Y-%m-%d_%H-%M-%S'
)
print(latest_path)

s3://data/student_performance/2026-08-12_21-51-22/


In [5]:
# Load data from bucket
X_train =  wr.s3.read_csv(latest_path+'X_train.csv')
y_train =  wr.s3.read_csv(latest_path+'y_train.csv')

X_test =  wr.s3.read_csv(latest_path+'X_test.csv')
y_test =  wr.s3.read_csv(latest_path+'y_test.csv')

In [ ]:
# Train and log a logistic regression
model_name = 'student_performance_logreg'
best_model, study, best_params = logistic_regression_model_train_log(
    X_train,
    y_train,
    X_test,
    y_test,
    experiment_name='Student Performance',
    model_name = model_name,
    n_trials=100,
    cv=5,
    scoring='f1',
    random_state=42
)

In [ ]:
# Register the model
client = MlflowClient()
latest_version = client.get_latest_versions(model_name)[-1].version
# Register as champion
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=latest_version
)

/tmp/ipykernel_464042/4187555006.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions(model_name)[-1].version
